# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ashishpal003/flyrank_ml_intern/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Connect to the warehouse

DuckDB reads `hf://` Parquet natively — only the columns and partitions a query touches are fetched. The `COUNT(*)` loop below touches Parquet **metadata**, not data, so it returns in seconds and confirms we are pointed at the right release.

In [1]:
%pip -q install duckdb huggingface_hub

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, getpass
from pathlib import Path

# Token order: env var -> .env file (local runs) -> Colab Secret -> prompt (last resort).
# Never commit the token: `.env` is gitignored and this repo is public.
def _from_dotenv(key):
    for base in [Path.cwd(), *Path.cwd().parents]:
        f = base / ".env"
        if f.is_file():
            for line in f.read_text().splitlines():
                s = line.strip()
                if s.startswith(f"{key}=") or s.startswith(f"export {key}="):
                    return s.split("=", 1)[1].strip().strip('"').strip("'")
    return None

HF_TOKEN = os.environ.get("HF_TOKEN") or _from_dotenv("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")
assert HF_TOKEN and HF_TOKEN.startswith("hf_"), "no valid HF READ token found (env / .env / Colab secret)"


In [3]:
import duckdb, json
from pathlib import Path

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")   # quiet output in nbconvert / no-widget runs
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':   f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':   f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':    f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d':f"read_parquet('{REL}/fact_content_query_90d.parquet')",
    # fact_content_daily_performance_sample (June 2026) is the SEALED test month -- not loaded here.
}

EXPECTED_ROWS = {'dim_clients': 104, 'dim_content': 519_606,
                 'fact_daily': 78_835_655, 'fact_query_90d': 2_414_248}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    exp = EXPECTED_ROWS[name]
    print(f"{name:16} {n:>12,} rows   (data-dictionary: {exp:>12,})   {'OK' if n == exp else 'MISMATCH'}")

# One decision date is used for every verification query in this notebook.
D = '2026-03-01'                       # decision_date
FEATURE_START = '2025-12-01'           # d - 90d  (feature window is [FEATURE_START, D))
LABEL_END = '2026-03-31'              # d + 30d  (label window is [D, LABEL_END])
FEATURE_MONTHS = ['2025-12', '2026-01', '2026-02']
LABEL_MONTHS = ['2026-03']

def daily(months):
    """read_parquet over an explicit list of month partitions (hf:// has no brace globs)."""
    paths = [f"'{REL}/fact_content_daily_performance/month={m}/*.parquet'" for m in months]
    return f"read_parquet([{', '.join(paths)}])"

print(f"\ndecision_date D = {D}   feature window [{FEATURE_START}, {D})   label window [{D}, {LABEL_END}]")


dim_clients               104 rows   (data-dictionary:          104)   OK


dim_content           519,606 rows   (data-dictionary:      519,606)   OK


fact_daily         78,835,655 rows   (data-dictionary:   78,835,655)   OK


fact_query_90d      2,414,248 rows   (data-dictionary:    2,414,248)   OK

decision_date D = 2026-03-01   feature window [2025-12-01, 2026-03-01)   label window [2026-03-01, 2026-03-31]


## 1. Unit of analysis + time window

**One row = one `(content_hash_id × decision_date)`** — one content page evaluated at one monthly decision date. (On the starter CSV this was one row per page, at a single implicit decision date = export time. The warehouse turns that into a panel: the same page reappears at each monthly decision date, each time with its own 90-day history behind it and its own 30-day outcome ahead of it.)

**Time windows** (strictly non-overlapping — nothing in the features can see the label):

```
          feature window                 |     label window
  [ decision_date - 90d , decision_date ) | [ decision_date , decision_date + 30d ]
  <---------- signals ------------------> | <-------- outcome ------->
```

**This notebook verifies everything at one decision date: `D = 2026-03-01`.** Feature window `[2025-12-01, 2026-03-01)` = partitions `2025-12` / `2026-01` / `2026-02`; label window `[2026-03-01, 2026-03-31]` = partition `2026-03`. All mid-panel. The **June 2026** partition is the sealed test month and is never read here.

**Per-client, not one global calendar cut.** `dim_clients.gsc_data_start` differs per client (unbalanced panel). Eligibility requires ≥ 90 days of history *before* `D`, checked against each client's own start date.

The cell below confirms: the fact table's date bounds, that the grain holds (no duplicate `report_date × client × content`), and that per-client histories do not start together.

In [4]:
# Live schema first -- the authoritative column list (adapt names below if these differ).
print('fact_content_daily_performance columns:')
print(con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_daily']}").df()[['column_name', 'column_type']].to_string(index=False))

bounds = con.sql(f"""
    SELECT MIN(report_date) AS min_date, MAX(report_date) AS max_date, COUNT(*) AS rows
    FROM {TABLES['fact_daily']}
""").df()
print('\ndate bounds / rows (expect 2025-01-27 / 2026-06-30 / 78,835,655):')
print(bounds.to_string(index=False))

# Grain probe on ONE partition (grain is uniform across partitions) -- expect zero rows.
grain = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {daily(['2026-03'])}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print(f'\ngrain probe (report_date x client x content), rows returned = {len(grain)}  ->  {"grain holds" if len(grain) == 0 else "GRAIN BROKEN"}')

# Per-client history does not start together.
spans = con.sql(f"""
    SELECT client_hash_id, MIN(report_date) AS first_seen, MAX(report_date) AS last_seen, COUNT(*) AS rows
    FROM {daily(['2026-02'])}
    GROUP BY 1 ORDER BY rows DESC LIMIT 10
""").df()
print('\nper-client row counts in one month partition (depth varies wildly):')
print(spans.to_string(index=False))

fact_content_daily_performance columns:


             column_name column_type
             report_date        DATE
          client_hash_id     VARCHAR
         content_hash_id     VARCHAR
          client_has_gsc     BOOLEAN
          client_has_ga4     BOOLEAN
      gsc_data_available     BOOLEAN
      ga4_data_available     BOOLEAN
         gsc_impressions      BIGINT
              gsc_clicks      BIGINT
        gsc_sum_position      BIGINT
        gsc_avg_position      DOUBLE
           ga4_pageviews      BIGINT
            ga4_sessions      BIGINT
               ga4_users      BIGINT
    ga4_engaged_sessions      BIGINT
ga4_total_engagement_sec      BIGINT
        sessions_organic      BIGINT
         sessions_direct      BIGINT
       sessions_referral      BIGINT
         sessions_social      BIGINT
           sessions_paid      BIGINT
             sessions_ai      BIGINT
              ai_chatgpt      BIGINT
           ai_perplexity      BIGINT
               ai_gemini      BIGINT
              ai_copilot      BIGINT
 


date bounds / rows (expect 2025-01-27 / 2026-06-30 / 78,835,655):
  min_date   max_date     rows
2025-01-27 2026-06-30 78835655



grain probe (report_date x client x content), rows returned = 0  ->  grain holds



per-client row counts in one month partition (depth varies wildly):
         client_hash_id first_seen  last_seen   rows
client_625b6439094e23e4 2026-02-01 2026-02-28 892836
client_08a6a72ff48e62c0 2026-02-01 2026-02-28 737194
client_73cda7b4e4f265ea 2026-02-01 2026-02-28 717712
client_62f4a7e64f5e0096 2026-02-01 2026-02-28 547865
client_65de48885f4ef01b 2026-02-01 2026-02-28 383082
client_ba65e80a1116ae41 2026-02-01 2026-02-28 370692
client_3ffa76342f366962 2026-02-01 2026-02-28 329756
client_23a62021009f63c4 2026-02-01 2026-02-28 319604
client_fef1a8f436438636 2026-02-01 2026-02-28 270677
client_3197e6291363b4db 2026-02-01 2026-02-28 264320


## 2. Fields: feature / label / context / excluded

Every column of all four tables, sorted into **exactly one** bucket. Column names are the **live schema** (`cell-4` prints them and asserts nothing is left unclassified).

| Bucket | Columns | Why |
|---|---|---|
| **Feature** — knowable *before* `D`, used only as an aggregate over `[D-90d, D)` | `dim_content`: `content_type`, `main_intent`, `competition_level`, `search_volume`, `competition`, `cpc`, `backlinks`, `category_count`, `word_count`, `char_count`, `keyword_char_count`, `keyword_token_count`, `url_char_count` (+ `content_age_days` / `days_since_update` **derived** from `content_created_date` / `content_updated_date` as of `D`). `fact_daily` (GA4 columns only where `ga4_data_available IS TRUE`): `gsc_impressions`, `gsc_clicks`, `gsc_sum_position`, `gsc_avg_position`, `ga4_pageviews`, `ga4_sessions`, `ga4_users`, `ga4_engaged_sessions`, `ga4_total_engagement_sec`, `sessions_organic/_direct/_referral/_social/_paid/_ai`, `ai_chatgpt/_perplexity/_gemini/_copilot/_claude/_meta/_other`, `scroll_events` | all computed strictly inside the feature window |
| **Label / proxy** — the thing we predict | **derived**: sustained impressions decline over `[D, D+30d]` (label-window impressions ≥ ~25% below trailing-90d pace, still falling — not a one-day spike). Its source column is `gsc_impressions` **restricted to `[D, D+30d]`** — the same column is a *feature* over the feature window and *label* over the label window; the window is what decides | never used as a feature over the label window |
| **Context** — group / join / split / filter / read only | keys `client_hash_id`, `content_hash_id`, `keyword_hash_id`, `url_hash_id`, `query_hash_id`; `report_date`, `month`, `window_start`, `window_end`; per-client bookkeeping `gsc_data_start`, `ga4_data_start`, `access_profile`, `client_has_gsc`, `client_has_ga4`, `has_gsc_access`, `has_ga4_access`, `is_active`; content bookkeeping `is_published`, `is_deleted`, `content_created_date`, `content_updated_date`, `client_created_date`, `client_updated_date`, `keyword_created_date`, `optimization_eligible_date` | identifiers, dates that *feed* derived features, and eligibility flags — never learned from directly |
| **Excluded** — off-limits (one line each) | | |
| | `fact_content_daily_performance_sample` / any `month=2026-06*` | June 2026 = sealed test month; develop-on-test is the cardinal sin |
| | `gsc_data_available`, `ga4_data_available` | row-level **three-valued** gate (TRUE / FALSE / NULL, see §3.5) — filter with `IS TRUE` / `IS NOT TRUE`, not a signal to learn |
| | **every non-key column of `fact_content_query_90d`** (`impressions_90d`, `clicks_90d`, `*_last30`, `*_prev30`, `avg_position_*`, `content_total_impressions_90d`, `content_visible_query_count`, `rare_query_count`, `rare_impressions_share`, `anonymized_impressions_share`, `query_char_count`, `query_token_count`) | the table has ONE fixed 90-day window `[2026-04-02, 2026-06-30]` (§3.6) — it ends on the panel's last day and sits entirely *after* any dev decision date, overlapping every possible label window. Not leakage-safe for this past→future lane. Query-mix features are out of scope here. |
| | `last_optimized_date` | FlyRank's own optimization action, tracked only from ~2026-04 (§3.6) so unknown as of an earlier `D`; and an optimization *inside* the label window would leak. Use `content_updated_date` for freshness instead. |
| | `provider_used`, `model_used` | which LLM generated the article — generation metadata, not an earned search signal |
| | any `ga4_*` value where `ga4_data_available IS NOT TRUE` | zero-filled / NULL placeholder — not a real zero (§3.5) |
| | raw query / URL / title / client text | scrambled before we got them; not in the data — if anything resembling one appears, stop and remove it |

In [5]:
# Pull the live column list of every table and check the contract covers all of it.
live_cols = {}
for name, src in TABLES.items():
    cols = con.sql(f"DESCRIBE SELECT * FROM {src}").df()['column_name'].tolist()
    live_cols[name] = cols
    print(f"{name:16} ({len(cols)} cols): {', '.join(cols)}\n")

# --- Contract buckets (live-schema names) ---
KEYS = {'client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'query_hash_id'}

FEATURE = {
    # dim_content, static / knowable at D
    'content_type', 'main_intent', 'competition_level', 'search_volume', 'competition', 'cpc',
    'backlinks', 'category_count', 'word_count', 'char_count',
    'keyword_char_count', 'keyword_token_count', 'url_char_count',
    # fact_daily, aggregated over [D-90d, D)  (GA4 cols only where ga4_data_available IS TRUE)
    'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position',
    'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec',
    'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid',
    'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude',
    'ai_meta', 'ai_other', 'scroll_events',
}
# LABEL is derived (built in cell-6d), not a stored column; its source is gsc_impressions over [D, D+30d].

CONTEXT = {
    'report_date', 'month', 'window_start', 'window_end',
    'gsc_data_start', 'ga4_data_start', 'access_profile',
    'client_has_gsc', 'client_has_ga4', 'has_gsc_access', 'has_ga4_access', 'is_active',
    'is_published', 'is_deleted',
    'content_created_date', 'content_updated_date', 'client_created_date', 'client_updated_date',
    'keyword_created_date', 'optimization_eligible_date',
}

EXCLUDED = {
    'gsc_data_available', 'ga4_data_available',        # three-valued gate -- filter, don't learn
    'provider_used', 'model_used',                     # generation metadata
    'last_optimized_date',                             # FlyRank action; sparse pre-2026-04; edit-in-label-window leaks
    # every non-key column of fact_content_query_90d -- its single window [2026-04-02, 2026-06-30]
    # overlaps every possible label window in this past->future lane (see 3.6):
    'query_char_count', 'query_token_count', 'impressions_90d', 'clicks_90d',
    'impressions_last30', 'clicks_last30', 'impressions_prev30', 'clicks_prev30',
    'avg_position_90d', 'avg_position_last30', 'avg_position_prev30',
    'content_total_impressions_90d', 'content_visible_query_count', 'rare_query_count',
    'rare_impressions_share', 'anonymized_impressions_share',
}

classified = KEYS | FEATURE | CONTEXT | EXCLUDED
all_live = sorted({c for cols in live_cols.values() for c in cols})
unclassified = [c for c in all_live if c not in classified]
overlap = sorted((FEATURE & CONTEXT) | (FEATURE & EXCLUDED) | (CONTEXT & EXCLUDED))

print('UNCLASSIFIED columns (must be empty):', unclassified or 'none -- contract covers every column')
print('DOUBLE-CLASSIFIED columns (must be empty):', overlap or 'none -- buckets are disjoint')
assert not unclassified and not overlap, 'contract is incomplete or inconsistent'


dim_clients      (9 cols): client_hash_id, is_active, has_gsc_access, has_ga4_access, access_profile, client_created_date, client_updated_date, gsc_data_start, ga4_data_start



dim_content      (26 cols): client_hash_id, content_hash_id, keyword_hash_id, url_hash_id, keyword_char_count, keyword_token_count, url_char_count, content_created_date, content_updated_date, content_type, search_volume, competition, competition_level, cpc, main_intent, backlinks, category_count, keyword_created_date, provider_used, model_used, char_count, word_count, last_optimized_date, optimization_eligible_date, is_published, is_deleted



fact_daily       (31 cols): report_date, client_hash_id, content_hash_id, client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available, gsc_impressions, gsc_clicks, gsc_sum_position, gsc_avg_position, ga4_pageviews, ga4_sessions, ga4_users, ga4_engaged_sessions, ga4_total_engagement_sec, sessions_organic, sessions_direct, sessions_referral, sessions_social, sessions_paid, sessions_ai, ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other, scroll_events, month



fact_query_90d   (21 cols): client_hash_id, content_hash_id, query_hash_id, query_char_count, query_token_count, window_start, window_end, impressions_90d, clicks_90d, impressions_last30, clicks_last30, impressions_prev30, clicks_prev30, avg_position_90d, avg_position_last30, avg_position_prev30, content_total_impressions_90d, content_visible_query_count, rare_query_count, rare_impressions_share, anonymized_impressions_share

UNCLASSIFIED columns (must be empty): none -- contract covers every column
DOUBLE-CLASSIFIED columns (must be empty): none -- buckets are disjoint


## 3. Verify it with queries (grain, counts, missing values, windows)

A contract claim without a query beside it is a guess. Each block below checks one claim from §1–§2, then §3.10 writes the numbers to `work/outputs/ml04_contract_checks.json` — the receipt the report traces back to.

In [6]:
checks = {'decision_date': D, 'feature_window': [FEATURE_START, D], 'label_window': [D, LABEL_END]}

# --- 3.1 counts vs the data dictionary (from the preamble loop) ---
row_counts = {name: con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0] for name, src in TABLES.items()}
checks['row_counts'] = row_counts
checks['row_counts_match'] = all(row_counts[k] == EXPECTED_ROWS[k] for k in row_counts)
print('3.1 row counts match data dictionary:', checks['row_counts_match'])

# --- 3.2 grain holds (probe from cell-2, re-run compactly) ---
dup = con.sql(f"""
    SELECT COUNT(*) FROM (
        SELECT 1 FROM {daily(['2026-03'])}
        GROUP BY report_date, client_hash_id, content_hash_id HAVING COUNT(*) > 1
    )
""").fetchone()[0]
checks['grain_duplicate_keys'] = dup
print('3.2 duplicate (date x client x content) keys:', dup, '-> grain holds' if dup == 0 else '-> GRAIN BROKEN')

# --- 3.3 windows: overall bounds + unbalanced panel ---
b = con.sql(f"SELECT MIN(report_date) mn, MAX(report_date) mx FROM {TABLES['fact_daily']}").df().iloc[0]
checks['report_date_min'] = str(b.mn); checks['report_date_max'] = str(b.mx)
panel = con.sql(f"""
    SELECT COUNT(*) AS clients,
           COUNT(*) FILTER (WHERE gsc_data_start >  DATE '{FEATURE_START}') AS lt_90d_before_D,
           COUNT(*) FILTER (WHERE gsc_data_start >  DATE '2025-12-31')      AS lt_6_months_total
    FROM {TABLES['dim_clients']}
""").df().iloc[0].to_dict()
checks['panel'] = {k: int(v) for k, v in panel.items()}
print('3.3 date bounds:', checks['report_date_min'], '->', checks['report_date_max'])
print('    clients:', panel['clients'], '| starting <90d before D:', panel['lt_90d_before_D'],
      '| <~6 months history total:', panel['lt_6_months_total'])

3.1 row counts match data dictionary: True


3.2 duplicate (date x client x content) keys: 0 -> grain holds


3.3 date bounds: 2025-01-27 00:00:00 -> 2026-06-30 00:00:00
    clients: 104 | starting <90d before D: 28 | <~6 months history total: 27


In [7]:
# --- 3.4 missingness is patterned, not random (grouped by content_type) ---
miss = con.sql(f"""
    SELECT content_type,
           COUNT(*) AS n,
           ROUND(AVG(CASE WHEN search_volume IS NULL THEN 1.0 ELSE 0 END), 3) AS null_search_volume,
           ROUND(AVG(CASE WHEN word_count   IS NULL THEN 1.0 ELSE 0 END), 3) AS null_word_count
    FROM {TABLES['dim_content']}
    GROUP BY content_type ORDER BY n DESC
""").df()
print('3.4 missingness by content_type (a blind fillna(0) would encode content_type):')
print(miss.to_string(index=False))
checks['missingness_by_content_type'] = miss.to_dict(orient='records')

# --- 3.5 availability flags are THREE-valued (TRUE / FALSE / NULL) ---
flags = con.sql(f"""
    SELECT ga4_data_available, COUNT(*) AS rows
    FROM {daily(['2026-03'])}
    GROUP BY 1 ORDER BY rows DESC
""").df()
flags['ga4_data_available'] = flags['ga4_data_available'].astype('string').fillna('NULL')
print('\n3.5 ga4_data_available in month=2026-03 (note the NULL row -> use IS TRUE / IS NOT TRUE, never = FALSE):')
print(flags.to_string(index=False))
checks['ga4_flag_distribution'] = dict(zip(flags['ga4_data_available'], flags['rows'].astype(int)))

3.4 missingness by content_type (a blind fillna(0) would encode content_type):
      content_type      n  null_search_volume  null_word_count
   keyword article 459174               0.186            0.381
    feedly article  57024               1.000            0.051
comparison article   3408               0.001            0.001



3.5 ga4_data_available in month=2026-03 (note the NULL row -> use IS TRUE / IS NOT TRUE, never = FALSE):
ga4_data_available    rows
             False 6408671
              NULL 3018741
              True  413966


In [8]:
# --- 3.6 query table window + optimization-date coverage: why both are excluded here ---
qwin = con.sql(f"""
    SELECT MIN(window_start) AS win_start, MAX(window_end) AS win_end,
           COUNT(DISTINCT window_start) AS distinct_windows,
           COUNT(*) AS rows, COUNT(DISTINCT content_hash_id) AS content_items
    FROM {TABLES['fact_query_90d']}
""").df().iloc[0].to_dict()
checks['query_table'] = {k: (str(v) if 'win' in k else int(v)) for k, v in qwin.items()}
print('3.6 fact_content_query_90d:', checks['query_table'])
print(f"    ONE fixed window {qwin['win_start']} -> {qwin['win_end']}; it ends on the panel's last day and")
print(f"    lies entirely AFTER D={D} (and its *_prev30 cols ~= late May, still after D). Even for the")
print( "    latest usable dev decision date it overlaps the label window -> excluded as a feature source here.")

optcov = con.sql(f"""
    SELECT ROUND(100.0 * COUNT(*) FILTER (WHERE last_optimized_date < DATE '{D}') / COUNT(*), 2) AS pct_known_at_D,
           MIN(last_optimized_date) AS earliest
    FROM {TABLES['dim_content']}
""").df().iloc[0].to_dict()
checks['last_optimized_known_at_D_pct'] = float(optcov['pct_known_at_D'])
print(f"\n    last_optimized_date known as of D: {optcov['pct_known_at_D']}% (earliest {optcov['earliest']})",
      "-> use content_updated_date for freshness, not last_optimized_date")

# --- 3.7 content that first appears partway through the feature window ---
reg = con.sql(f"""
    WITH first_row AS (
        SELECT content_hash_id, MIN(report_date) AS first_seen
        FROM {daily(FEATURE_MONTHS)}
        GROUP BY 1
    )
    SELECT COUNT(*) AS content_in_window,
           COUNT(*) FILTER (WHERE first_seen > DATE '{FEATURE_START}') AS first_seen_after_start
    FROM first_row
""").df().iloc[0].to_dict()
reg['pct_after_start'] = round(100 * reg['first_seen_after_start'] / max(reg['content_in_window'], 1), 2)
checks['registration'] = {k: (int(v) if k != 'pct_after_start' else v) for k, v in reg.items()}
print(f"\n3.7 content items whose first row lands after the feature-window start (new / late-onboarded):",
      f"{reg['first_seen_after_start']:,} ({reg['pct_after_start']}%) -- for these, earlier history is unknown, not zero.")


3.6 fact_content_query_90d: {'win_start': '2026-04-02 00:00:00', 'win_end': '2026-06-30 00:00:00', 'distinct_windows': '1', 'rows': 2414248, 'content_items': 133852}
    ONE fixed window 2026-04-02 00:00:00 -> 2026-06-30 00:00:00; it ends on the panel's last day and
    lies entirely AFTER D=2026-03-01 (and its *_prev30 cols ~= late May, still after D). Even for the
    latest usable dev decision date it overlaps the label window -> excluded as a feature source here.



    last_optimized_date known as of D: 0.0% (earliest 2026-04-24 00:00:00) -> use content_updated_date for freshness, not last_optimized_date



3.7 content items whose first row lands after the feature-window start (new / late-onboarded): 72,851 (22.66%) -- for these, earlier history is unknown, not zero.


In [9]:
# --- 3.8 + 3.9 eligibility funnel and one illustrative labeled slice at D = 2026-03-01 ---
slice_df = con.sql(f"""
    WITH feat AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(f.gsc_impressions)                                                          AS imp_90d,
               COUNT(DISTINCT CASE WHEN f.gsc_impressions > 0 THEN f.report_date END)           AS days_with_impr_90d,
               SUM(CASE WHEN f.report_date >= DATE '{D}' - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <  DATE '{D}' - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev60
        FROM {daily(FEATURE_MONTHS)} f
        WHERE f.report_date >= DATE '{FEATURE_START}' AND f.report_date < DATE '{D}'
        GROUP BY 1, 2
    ),
    lab AS (
        SELECT content_hash_id,
               SUM(gsc_impressions)                                                            AS imp_label,
               SUM(CASE WHEN report_date <  DATE '{D}' + INTERVAL 15 DAY THEN gsc_impressions ELSE 0 END) AS imp_label_h1,
               SUM(CASE WHEN report_date >= DATE '{D}' + INTERVAL 15 DAY THEN gsc_impressions ELSE 0 END) AS imp_label_h2
        FROM {daily(LABEL_MONTHS)}
        WHERE report_date >= DATE '{D}' AND report_date <= DATE '{LABEL_END}'
        GROUP BY 1
    ),
    joined AS (
        SELECT feat.*, c.gsc_data_start,
               COALESCE(lab.imp_label, 0)    AS imp_label,
               COALESCE(lab.imp_label_h1, 0) AS imp_label_h1,
               COALESCE(lab.imp_label_h2, 0) AS imp_label_h2,
               feat.imp_90d / 90.0 * 30.0    AS expected_30d
        FROM feat
        LEFT JOIN lab ON feat.content_hash_id = lab.content_hash_id
        LEFT JOIN {TABLES['dim_clients']} c ON feat.client_hash_id = c.client_hash_id
    )
    SELECT *,
           (imp_last30 >= 100)                                              AS pass_volume_floor,
           (gsc_data_start <= DATE '{D}' - INTERVAL 90 DAY)                  AS pass_client_history,
           NOT (imp_last30 < 0.5 * NULLIF(imp_prev60, 0))                    AS pass_not_freefall,
           ( imp_label < 0.75 * expected_30d AND imp_label_h2 <= imp_label_h1 ) AS label_decline
    FROM joined
""").df()

total = len(slice_df)
f1 = slice_df['pass_volume_floor']
f2 = f1 & slice_df['pass_client_history'].fillna(False)
f3 = f2 & slice_df['pass_not_freefall'].fillna(True)
eligible = slice_df[f3].copy()
funnel = {'content_in_feature_window': total,
          'after_volume_floor(imp_last30>=100)': int(f1.sum()),
          'after_client_history(>=90d before D)': int(f2.sum()),
          'after_exclude_freefall': int(f3.sum())}
checks['eligibility_funnel'] = funnel
checks['label_base_rate'] = round(float(eligible['label_decline'].mean()), 4)

print('3.8 eligibility funnel at D =', D)
for k, v in funnel.items():
    print(f'    {k:38} {v:>8,}')
print(f"\n3.9 illustrative labeled slice: {len(eligible):,} rows, one per (content x D={D}); "
      f"label_decline base rate = {checks['label_base_rate']:.3f}")
eligible[['client_hash_id', 'content_hash_id', 'imp_90d', 'days_with_impr_90d',
          'imp_last30', 'expected_30d', 'imp_label', 'label_decline']].head(8)

3.8 eligibility funnel at D = 2026-03-01
    content_in_feature_window               321,546
    after_volume_floor(imp_last30>=100)      81,521
    after_client_history(>=90d before D)     73,326
    after_exclude_freefall                   60,265

3.9 illustrative labeled slice: 60,265 rows, one per (content x D=2026-03-01); label_decline base rate = 0.072


,client_hash_id,content_hash_id,imp_90d,days_with_impr_90d,imp_last30,expected_30d,imp_label,label_decline
5,client_73cda7b4e4f265ea,content_8b3eb19ed7281644,1144.0,90,499.0,381.333333,432.0,False
6,client_73cda7b4e4f265ea,content_7f0c5be4a936e3ea,2063.0,90,1023.0,687.666667,1232.0,False
8,client_73cda7b4e4f265ea,content_72e574fc6d247af2,48690.0,90,16549.0,16230.000000,19556.0,False
9,client_73cda7b4e4f265ea,content_3b0c75f1a15c23b2,8040.0,90,3663.0,2680.000000,4903.0,False
11,client_73cda7b4e4f265ea,content_27b2acaf093425c7,1052.0,90,498.0,350.666667,820.0,False
12,client_73cda7b4e4f265ea,content_d30379951f594549,7011.0,90,2665.0,2337.000000,3196.0,False
13,client_73cda7b4e4f265ea,content_60cbb20caaf7ceb3,3698.0,90,2427.0,1232.666667,2437.0,False
14,client_73cda7b4e4f265ea,content_6cd1d02a0d5cc858,13110.0,90,5578.0,4370.000000,6273.0,False


In [10]:
# --- 3.10 write the receipt (committed; the report's numbers trace back to this) ---
OUT = None
for cand in [Path('work/outputs'), Path('../outputs'), Path('outputs')]:
    if cand.parent.exists():
        OUT = cand
        break
OUT = OUT or Path('work/outputs')
OUT.mkdir(parents=True, exist_ok=True)

receipt_path = OUT / 'ml04_contract_checks.json'
receipt_path.write_text(json.dumps(checks, indent=2, default=str))
print('wrote', receipt_path.resolve())
print(json.dumps(checks, indent=2, default=str))

wrote /Users/ashishpal/Documents/flyrank/flyrank_ml_intern/work/outputs/ml04_contract_checks.json
{
  "decision_date": "2026-03-01",
  "feature_window": [
    "2025-12-01",
    "2026-03-01"
  ],
  "label_window": [
    "2026-03-01",
    "2026-03-31"
  ],
  "row_counts": {
    "dim_clients": 104,
    "dim_content": 519606,
    "fact_daily": 78835655,
    "fact_query_90d": 2414248
  },
  "row_counts_match": true,
  "grain_duplicate_keys": 0,
  "report_date_min": "2025-01-27 00:00:00",
  "report_date_max": "2026-06-30 00:00:00",
  "panel": {
    "clients": 104,
    "lt_90d_before_D": 28,
    "lt_6_months_total": 27
  },
  "missingness_by_content_type": [
    {
      "content_type": "keyword article",
      "n": 459174,
      "null_search_volume": 0.186,
      "null_word_count": 0.381
    },
    {
      "content_type": "feedly article",
      "n": 57024,
      "null_search_volume": 1.0,
      "null_word_count": 0.051
    },
    {
      "content_type": "comparison article",
      "n": 3408,

## 4. Data limits

What this data can **never** tell us — and the constraints ML-05 inherits:

- **Unbalanced panel.** Per-client GSC history runs from ~3 to ~17 months; ~26% of clients have under ~6 months (§3.3, §4 cell). Any single global calendar window, and any raw cross-client comparison, is unsafe — use per-client windows and client-grouped splits.
- **GSC-only / no-GA4 majority.** In `month=2026-03`, ~96% of rows have `ga4_data_available IS NOT TRUE` (§3.5 / §4 cell) — GA4 engagement columns there are zero-filled or NULL placeholders, not "no engagement". GA4 features are usable only on the thin `IS TRUE` slice; GSC (impressions / clicks / position) is the dependable signal.
- **No leakage-safe query-mix features here.** `fact_content_query_90d` has one fixed window `2026-04-02 → 2026-06-30` (§3.6). It ends on the panel's last day and lies entirely after any dev decision date, overlapping every possible 30-day label window — so query diversity / concentration / rare-tail signals are **out of scope for this past→future lane** (they'd be usable only for a decision date after the panel ends).
- **Freshness signal is thin early.** `last_optimized_date` is populated only from ~2026-04 (§3.6), so "days since last optimization" is unknown as of `D = 2026-03-01`; freshness must come from `content_updated_date`. And an optimization performed *inside* the label window would leak — it is on the red list.
- **Registration-day accrual.** ~23% of content in the feature window first appears partway through it (§3.7). Missing early rows mean *unknown*, not zero, traffic.
- **Pseudonymised, one operation, one 17-month snapshot.** Everything here is association, observed on FlyRank's own portfolio. No causal claims (seasonality, market, cannibalisation are uncontrolled), nothing about whether a refresh *works*, and no claim that any of it generalises beyond this operation or period.

**Output of the wider project (one sentence):** a weekly, per-client ranked queue of the pages most likely to lose search impressions over the next 30 days, each with reason codes — a decision aid for a FlyRank content strategist, evaluated against the base rate and FlyRank's current stale-visible rule.

In [11]:
limits = con.sql(f"""
    SELECT
        ROUND(100.0 * COUNT(*) FILTER (WHERE gsc_data_start > DATE '2025-12-31') / COUNT(*), 1) AS pct_clients_lt_6mo_history
    FROM {TABLES['dim_clients']}
""").df().iloc[0]['pct_clients_lt_6mo_history']

ga4_not_true = con.sql(f"""
    SELECT ROUND(100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS NOT TRUE) / COUNT(*), 1) AS pct
    FROM {daily(['2026-03'])}
""").df().iloc[0]['pct']

print(f'clients with < ~6 months of GSC history:            {limits}%')
print(f'month=2026-03 rows with ga4_data_available IS NOT TRUE: {ga4_not_true}%  (GA4 columns there are placeholders, not zeros)')

clients with < ~6 months of GSC history:            26.0%
month=2026-03 rows with ga4_data_available IS NOT TRUE: 95.8%  (GA4 columns there are placeholders, not zeros)


## Self-check

- [x] Every section above is filled — markdown reasoning AND a backing query cell
- [x] Runs top to bottom with no errors (verified via `nbconvert --execute` against the live warehouse; token from `.env`)
- [x] Every warehouse column lands in exactly one contract bucket — asserted in §2 (`cell-4`): 0 unclassified, 0 double-classified
- [x] The June 2026 partition / `fact_content_daily_performance_sample` is never queried — sealed test month
- [x] IDs shown are hashes; no client names, URLs, or raw queries anywhere
- [x] `trend_direction` / `trend_pct` / `is_declining_label` / product flags / query-90d columns appear only in the *Excluded* bucket
- [x] Claims use careful words: observed, measured, directional, decision-support
- [x] Receipt written to `work/outputs/ml04_contract_checks.json`
- [ ] Commit `work/notebooks/w03_data_contract.ipynb` (with outputs) + `work/outputs/ml04_contract_checks.json`

### Numbers this contract established (for ML-05 to build on)

| Claim | Verified value |
|---|---|
| warehouse row counts | 104 / 519,606 / 78,835,655 / 2,414,248 — all match the data dictionary |
| grain `report_date × client × content` | holds (0 duplicate keys) |
| panel dates | 2025-01-27 → 2026-06-30; ~26% of clients have < ~6 months history |
| GA4 availability in `month=2026-03` | ~96% of rows `ga4_data_available IS NOT TRUE` (FALSE **or** NULL) |
| missingness by `content_type` | patterned — `feedly article` = 100% null `search_volume` vs 18.6% for `keyword article` |
| `fact_content_query_90d` window | single fixed `2026-04-02 → 2026-06-30` → not leakage-safe here; query-mix features out of scope |
| `last_optimized_date` known at `D = 2026-03-01` | 0% (earliest 2026-04-24) → freshness from `content_updated_date` |
| eligibility funnel at `D` | 321,546 → 81,521 (vol. floor) → 73,326 (client history) → **60,265** eligible |
| forward 30-day sustained-decline label base rate | **~0.072** (vs the ML-03 within-snapshot proxy's 0.37 — this is the real, rare base rate) |